# Matched fixed versus multi-layout I-JEPA Wall experiment

Run cells in order on a Colab GPU runtime. This trains **two new models** from separate datasets: one uses only the default wall/door layout `[32, 30]`; the other uses nine training layouts. Both datasets use the same correlated-heading `wide` policy with uniformly sampled action lengths from 0.2 to 3.5. The validation and test layouts, seeds, encoder, model settings, and planner are matched.

Raw episodes, images, trajectories, and embeddings stay under `/content`. Drive receives only manifests, provenance, model checkpoints, logs, and result files. A runtime reset deletes local datasets; rerun the notebook to regenerate them before resuming a saved checkpoint. The dense cache is about 81 GiB per model, so the notebook clears the first model's local files before generating the second dataset. A full run can take many hours and may need a Colab runtime with substantial local disk.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys, time
import numpy as np
import torch

REPO = Path("/content/i-jepa-wm")
DINO_ROOT = Path("/content/dino_wm")
BASE_COMMIT = "37a3e4e9b086319b050eb1dd3d3a2eb9e42c3eaa"
DINO_COMMIT = "0a9492fa12044b852ae9e001cc74604b79c8bb0c"
PATCH_SHA256 = "93185feda9a71e638b00751bf3abc3f456b3f04567b02573173c264c808ca742"
PATCH_B64 = "..."
PATCH = base64.b64decode(PATCH_B64)
assert hashlib.sha256(PATCH).hexdigest() == PATCH_SHA256

def shell(args, *, cwd=None):
    args = [str(arg) for arg in args]
    print("$", " ".join(args), flush=True)
    subprocess.run(args, cwd=cwd, check=True)

def run(args):
    shell([sys.executable, "-u", "-m", *args], cwd=REPO)

if not REPO.exists():
    shell(["git", "clone", "--branch", "wall-layout-generalization", "--single-branch",
           "https://github.com/kmylonas/i-jepa-wm.git", REPO])
marker = REPO / ".matched_wall_source_patch.sha256"
if marker.is_file():
    assert marker.read_text().strip() == PATCH_SHA256, "Different source patch already applied"
else:
    assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO).strip(), (
        "Existing checkout has edits; use a clean Colab runtime"
    )
    shell(["git", "checkout", BASE_COMMIT], cwd=REPO)
    patch_path = Path("/content/matched_wall_source.patch")
    patch_path.write_bytes(PATCH)
    shell(["git", "apply", "--check", patch_path], cwd=REPO)
    shell(["git", "apply", patch_path], cwd=REPO)
    marker.write_text(PATCH_SHA256)

if not DINO_ROOT.exists():
    shell(["git", "clone", "https://github.com/gaoyuezhou/dino_wm.git", DINO_ROOT])
shell(["git", "-C", DINO_ROOT, "checkout", DINO_COMMIT])
shell([sys.executable, "-m", "pip", "install", "-q",
       "gym==0.26.2", "omegaconf", "scipy", "pyyaml", "imageio", "imageio-ffmpeg"])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
assert torch.cuda.is_available(), "Select a GPU runtime before continuing"
print("GPU:", torch.cuda.get_device_name(0))
print("Pinned source:", BASE_COMMIT[:12], "patch:", PATCH_SHA256[:12])


In [ ]:
from src.mylonas_ijepa_wm.wall_layout_manifest import (
    TRAIN_LAYOUTS, load_manifest, manifest_digest, prepare_manifest,
    validate_checkpoint_manifest,
)
from src.mylonas_ijepa_wm.audit_wall_dataset import compare_action_hashes

DRIVE_ROOT = Path("/content/drive/MyDrive/datasets/wall_matched_fixed_multi_v1")
LOCAL_ROOT = Path("/content/wall_matched_fixed_multi_v1")
SOURCE_CANDIDATES = [
    Path("/content/drive/MyDrive/checkpoints/IN1K-vit.h.14-300e.pth.tar"),
    Path("/content/drive/MyDrive/checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar"),
]
SOURCE_DRIVE = next((path for path in SOURCE_CANDIDATES if path.is_file()), None)
assert SOURCE_DRIVE is not None, f"I-JEPA checkpoint missing; checked {SOURCE_CANDIDATES}"
ENCODER_LOCAL = LOCAL_ROOT / "ijepa_encoder.pth.tar"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
assert LOCAL_ROOT.resolve().is_relative_to(Path("/content"))
assert not LOCAL_ROOT.resolve().is_relative_to(Path("/content/drive"))
if not ENCODER_LOCAL.is_file() or ENCODER_LOCAL.stat().st_size != SOURCE_DRIVE.stat().st_size:
    assert shutil.disk_usage("/content").free > SOURCE_DRIVE.stat().st_size + 5 * 2**30
    shutil.copy2(SOURCE_DRIVE, ENCODER_LOCAL)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 2**20), b""):
            digest.update(block)
    return digest.hexdigest()

ENCODER_SHA256 = sha256_file(ENCODER_LOCAL)
assert ENCODER_SHA256 == sha256_file(SOURCE_DRIVE)
print("Encoder:", SOURCE_DRIVE)
print("Free runtime disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

MODEL_DIRS = {mode: DRIVE_ROOT / mode for mode in ("fixed", "multi")}
for mode, model_drive in MODEL_DIRS.items():
    model_drive.mkdir(parents=True, exist_ok=True)
    manifest_path = model_drive / "episode_manifest.json"
    config_path = model_drive / "environment.json"
    expected = prepare_manifest(train_layout_mode=mode, collection_profile="wide_only")
    if not manifest_path.exists() and not config_path.exists():
        run(["src.mylonas_ijepa_wm.wall_layout_manifest",
             "--output-dir", model_drive, "--dino-wm-root", DINO_ROOT,
             "--train-layout-mode", mode, "--collection-profile", "wide_only"])
    assert manifest_path.is_file() and config_path.is_file()
    assert load_manifest(manifest_path) == expected, f"Existing {mode} manifest differs"
    settings = json.loads(config_path.read_text())
    assert settings["dino_wm_root"] == str(DINO_ROOT)
    assert settings["wall_layouts"] == [*expected["train_layouts"],
                                         *expected["val_layouts"], *expected["test_layouts"]]

fixed_manifest = load_manifest(MODEL_DIRS["fixed"] / "episode_manifest.json")
multi_manifest = load_manifest(MODEL_DIRS["multi"] / "episode_manifest.json")
assert fixed_manifest["train_layouts"] == [[32, 30]]
assert multi_manifest["train_layouts"] == TRAIN_LAYOUTS
for split in ("val", "test"):
    assert [e for e in fixed_manifest["episodes"] if e["split"] == split] == [
        e for e in multi_manifest["episodes"] if e["split"] == split]
print("Fixed train layouts:", fixed_manifest["train_layouts"])
print("Multi train layouts:", multi_manifest["train_layouts"])
print("Validation and test assignments match")


In [ ]:
EPISODES = 2600
RAW_EPISODE_BYTES = 51 * 224 * 224 * 3
LATENT_EPISODE_BYTES = 51 * 256 * 1280 * 2
BATCH = 100

def model_paths(mode):
    model_drive = MODEL_DIRS[mode]
    local_model_root = LOCAL_ROOT / mode
    assert local_model_root.resolve().is_relative_to(LOCAL_ROOT.resolve())
    return dict(
        local_model_root=local_model_root,
        raw=local_model_root / "raw", cache=local_model_root / "cache_dense",
        drive=model_drive, manifest=model_drive / "episode_manifest.json",
        env=model_drive / "environment.json",
        checkpoint_dir=model_drive / "world_model", results=model_drive / "results",
    )

def manifest_for(mode):
    return fixed_manifest if mode == "fixed" else multi_manifest

def episode_files(paths, index):
    stem = f"episode_{index:04d}"
    return (paths["raw"] / f"{stem}.pt",
            paths["cache"] / "latents" / f"{stem}.npy",
            paths["cache"] / "trajectories" / f"{stem}.pt")

def valid_cache_pair(paths, manifest, index):
    _, latent_path, trajectory_path = episode_files(paths, index)
    if not latent_path.is_file() or not trajectory_path.is_file():
        return False
    latent = np.load(latent_path, mmap_mode="r")
    if latent.shape != (51, 256, 1280) or latent.dtype != np.float16:
        return False
    trajectory = torch.load(trajectory_path, map_location="cpu", weights_only=True)
    assignment = manifest["episodes"][index]
    return (trajectory.get("manifest_sha256") == manifest_digest(manifest)
            and trajectory.get("seed") == assignment["seed"]
            and list(trajectory.get("wall_layout")) == assignment["wall_layout"]
            and trajectory.get("embedding_frame_stride") == 1)

def run_model(mode):
    assert mode in ("fixed", "multi")
    if mode == "multi":
        assert not model_paths("fixed")["local_model_root"].exists(), (
            "Clear verified fixed runtime data before generating the multi dataset")
    paths = model_paths(mode)
    manifest = manifest_for(mode)
    paths["raw"].mkdir(parents=True, exist_ok=True)
    paths["cache"].mkdir(parents=True, exist_ok=True)
    paths["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
    paths["results"].mkdir(parents=True, exist_ok=True)
    if (paths["drive"] / "complete.json").is_file():
        verify_model_outputs(mode)
        print(mode, "already complete; runtime data can be cleared")
        return

    audit_path = paths["results"] / "coverage.json"
    cached_count = sum(valid_cache_pair(paths, manifest, i) for i in range(EPISODES))
    raw_count = sum(episode_files(paths, i)[0].is_file() for i in range(EPISODES))
    if cached_count == 0 or not audit_path.is_file():
        missing = EPISODES - raw_count
        needed = missing * RAW_EPISODE_BYTES + 8 * 2**30
        free = shutil.disk_usage("/content").free
        print(mode, "raw episodes missing:", missing, "free GiB:", round(free / 2**30, 1))
        assert free > needed, "Insufficient /content disk for raw collection"
        run(["src.mylonas_ijepa_wm.generate_episodes",
             "--env", "wall", "--env-config", paths["env"],
             "--episode-manifest", paths["manifest"], "--output-dir", paths["raw"],
             "--steps-per-episode", "50", "--resume"])
        assert all(episode_files(paths, i)[0].is_file() for i in range(EPISODES))
        run(["src.mylonas_ijepa_wm.audit_wall_dataset",
             "--raw-dir", paths["raw"], "--episode-manifest", paths["manifest"],
             "--output", audit_path])
    audit = json.loads(audit_path.read_text())
    assert audit["manifest_sha256"] == manifest_digest(manifest)
    if not audit["coverage_pass"]:
        print(f"{mode} did not meet the old mixed-policy coverage threshold; "
              f"counts are saved at {audit_path}. Continuing with the predeclared wide-only policy.")
    if mode == "multi":
        fixed_audit = json.loads((MODEL_DIRS["fixed"] / "results/coverage.json").read_text())
        print("Matched actions:", compare_action_hashes(fixed_audit, audit)["matching"])

    raw_bytes = sum(path.stat().st_size for path in paths["raw"].glob("episode_*.pt"))
    free = shutil.disk_usage("/content").free
    assert free + raw_bytes > EPISODES * LATENT_EPISODE_BYTES + 10 * 2**30, (
        "Insufficient /content disk for dense embeddings even after raw batch deletion")
    for start in range(0, EPISODES, BATCH):
        end = min(start + BATCH, EPISODES)
        missing = [i for i in range(start, end) if not valid_cache_pair(paths, manifest, i)]
        if missing:
            run(["src.mylonas_ijepa_wm.generate_episodes",
                 "--env", "wall", "--env-config", paths["env"],
                 "--episode-manifest", paths["manifest"], "--output-dir", paths["raw"],
                 "--steps-per-episode", "50", "--start-index", start,
                 "--end-index", end, "--resume"])
            free = shutil.disk_usage("/content").free
            assert free > len(missing) * LATENT_EPISODE_BYTES + 2 * 2**30, (
                f"Insufficient runtime disk for embedding batch {start}:{end}")
            run(["src.mylonas_ijepa_wm.embed_frames",
                 "--env", "wall", "--env-config", paths["env"],
                 "--input-dir", paths["raw"], "--output-dir", paths["cache"],
                 "--checkpoint-path", ENCODER_LOCAL,
                 "--embedding-frame-stride", "1", "--start-index", start,
                 "--end-index", end, "--encoder-batch-size", "4", "--resume"])
        assert all(valid_cache_pair(paths, manifest, i) for i in range(start, end))
        for i in range(start, end):
            episode_files(paths, i)[0].unlink(missing_ok=True)
        print(mode, "verified and cleared raw batch", start, end, flush=True)
    assert all(valid_cache_pair(paths, manifest, i) for i in range(EPISODES))
    assert not list(paths["raw"].glob("episode_*.pt"))

    identity = dict(manifest_sha256=manifest_digest(manifest), encoder_sha256=ENCODER_SHA256,
                    source_base=BASE_COMMIT, source_patch_sha256=PATCH_SHA256,
                    policy="wide_only", action_min=0.2, action_max=3.5,
                    num_hist=1, rollout_steps=1, epochs=100, batch_size=8, seed=42)
    identity_path = paths["drive"] / "run_identity.json"
    if identity_path.exists():
        assert json.loads(identity_path.read_text()) == identity, "Existing training run differs"
    else:
        assert not (paths["checkpoint_dir"] / "last.pt").exists()
        identity_path.write_text(json.dumps(identity, indent=2))
    train = ["src.mylonas_ijepa_wm.wm_proprio",
             "--env", "wall", "--env-config", paths["env"],
             "--episode-manifest", paths["manifest"],
             "--latent-dir", paths["cache"] / "latents",
             "--trajectory-dir", paths["cache"] / "trajectories",
             "--output-dir", paths["checkpoint_dir"],
             "--num-hist", "1", "--rollout-steps", "1",
             "--epochs", "100", "--batch-size", "8",
             "--seed", "42", "--device", "cuda", "--precision", "auto",
             "--log-every", "1000"]
    if (paths["checkpoint_dir"] / "last.pt").is_file():
        train.append("--resume")
    run(train)
    best = paths["checkpoint_dir"] / "best.pt"
    assert best.is_file() and (paths["checkpoint_dir"] / "last.pt").is_file()
    metadata = torch.load(best, map_location="cpu", weights_only=True)["metadata"]
    validate_checkpoint_manifest(metadata, manifest)

    for split in ("val", "test"):
        run(["src.mylonas_ijepa_wm.evaluate_rollouts_proprio",
             "--checkpoint", best,
             "--latent-dir", paths["cache"] / "latents",
             "--trajectory-dir", paths["cache"] / "trajectories",
             "--split", split, "--horizon", "5", "--by-wall-layout",
             "--device", "cuda", "--precision", "auto",
             "--output", paths["results"] / f"{split}_rollouts_h5.json"])
    run(["src.mylonas_ijepa_wm.evaluate_wall_collision_probes",
         "--checkpoint", best, "--encoder-checkpoint", ENCODER_LOCAL,
         "--env-config", paths["env"], "--episode-manifest", paths["manifest"],
         "--device", "cuda", "--precision", "auto",
         "--output", paths["results"] / "heldout_collision_probes.json"])
    run(["src.mylonas_ijepa_wm.evaluate_wall_visual_pairs",
         "--checkpoint", best, "--checkpoint-role", mode,
         "--encoder-checkpoint", ENCODER_LOCAL,
         "--episode-manifest", paths["manifest"], "--env-config", paths["env"],
         "--device", "cuda", "--precision", "auto",
         "--output", paths["results"] / "heldout_visual_pairs.json"])
    sanity_env = paths["results"] / "default_layout_environment.json"
    sanity_settings = json.loads(paths["env"].read_text())
    sanity_settings.update(wall_location=32, door_location=30, wall_layouts=[[32, 30]])
    sanity_env.write_text(json.dumps(sanity_settings, indent=2))
    run(["src.mylonas_ijepa_wm.plan_proprio",
         "--env", "wall", "--env-config", sanity_env,
         "--encoder-checkpoint", ENCODER_LOCAL, "--world-model-checkpoint", best,
         "--output", paths["results"] / "default_layout_sanity.json",
         "--cost", "position", "--num-episodes", "20", "--base-seed", "10000",
         "--max-macro-steps", "20", "--planning-horizon", "5",
         "--actions-per-plan", "5", "--population", "300", "--num-elites", "30",
         "--cem-iterations", "10", "--candidate-batch-size", "64",
         "--unbounded-actions", "--max-action-length", "3.5",
         "--device", "cuda", "--precision", "auto"])
    verify_model_outputs(mode)
    complete = dict(identity=identity, best_sha256=sha256_file(best),
                    completed_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    (paths["drive"] / "complete.json").write_text(json.dumps(complete, indent=2))
    print(mode, "model complete and verified on Drive")

def verify_model_outputs(mode):
    paths = model_paths(mode)
    manifest = manifest_for(mode)
    best = paths["checkpoint_dir"] / "best.pt"
    assert best.is_file() and (paths["checkpoint_dir"] / "last.pt").is_file()
    metadata = torch.load(best, map_location="cpu", weights_only=True)["metadata"]
    validate_checkpoint_manifest(metadata, manifest)
    result_files = ["coverage.json", "val_rollouts_h5.json", "test_rollouts_h5.json",
                    "heldout_collision_probes.json", "heldout_visual_pairs.json",
                    "default_layout_sanity.json"]
    for name in result_files:
        path = paths["results"] / name
        assert path.is_file() and path.stat().st_size > 0, f"Missing result: {path}"
        json.loads(path.read_text())
    complete_path = paths["drive"] / "complete.json"
    if complete_path.is_file():
        complete = json.loads(complete_path.read_text())
        assert complete["identity"] == json.loads((paths["drive"] / "run_identity.json").read_text())
        assert complete["best_sha256"] == sha256_file(best)
    return True

def cleanup_model(mode):
    verify_model_outputs(mode)
    paths = model_paths(mode)
    local_model_root = paths["local_model_root"]
    if local_model_root.exists():
        assert local_model_root.resolve().is_relative_to(LOCAL_ROOT.resolve())
        shutil.rmtree(local_model_root)
    assert not local_model_root.exists()
    print("Cleared", mode, "runtime dataset and cache; free GiB:",
          round(shutil.disk_usage("/content").free / 2**30, 1))


## Small collection pilot

This short pilot checks the pinned environment, manifest assignments, action lengths, and contact counts before the full datasets are generated. Pilot files are removed from `/content` after the audit; they are not uploaded to Drive. Ten episodes per training layout are too few to satisfy the full coverage thresholds.

In [ ]:
def pilot_model(mode):
    pilot_root = LOCAL_ROOT / "pilot" / mode
    if pilot_root.exists():
        shutil.rmtree(pilot_root)
    train_count = 10 if mode == "fixed" else 90
    run(["src.mylonas_ijepa_wm.wall_layout_manifest",
         "--output-dir", pilot_root, "--dino-wm-root", DINO_ROOT,
         "--train-layout-mode", mode, "--collection-profile", "wide_only",
         "--base-seed", "100000", "--train-episodes", train_count,
         "--val-episodes", "2", "--test-episodes", "4"])
    run(["src.mylonas_ijepa_wm.generate_episodes",
         "--env", "wall", "--env-config", pilot_root / "environment.json",
         "--episode-manifest", pilot_root / "episode_manifest.json",
         "--output-dir", pilot_root / "raw", "--steps-per-episode", "50"])
    run(["src.mylonas_ijepa_wm.audit_wall_dataset",
         "--raw-dir", pilot_root / "raw",
         "--episode-manifest", pilot_root / "episode_manifest.json",
         "--output", pilot_root / "coverage.json"])
    audit = json.loads((pilot_root / "coverage.json").read_text())
    for layout in manifest_for(mode)["train_layouts"]:
        row = audit["layouts"][f"{layout[0]},{layout[1]}"]
        print(mode, layout, "contacts:", row["solid_contacts"],
              "door crossings:", row["door_crossings_left_to_right"] + row["door_crossings_right_to_left"],
              "actions >3.0:", row["length_gt_3_0"])
    shutil.rmtree(pilot_root)
    print("Cleared", mode, "pilot runtime files")

pilot_model("fixed")
pilot_model("multi")


## Train the two new checkpoints

The first model's raw episodes and dense cache are deleted only after its Drive checkpoint and result files have been verified. The multi-layout dataset is generated after that deletion. The old mixed-policy coverage thresholds are reported but do not stop this correlated-only experiment; inspect `coverage.json` when interpreting the results.


In [ ]:
run_model("fixed")
cleanup_model("fixed")
run_model("multi")
cleanup_model("multi")


## Matched held-out planning

This runs 50 paired seeds on each of four unseen layouts, with the same position-cost CEM settings and a primitive-action cap of 3.5. It stores per-run JSON and a paired bootstrap summary on Drive. It can resume partial planning results.


In [ ]:
verify_model_outputs("fixed")
verify_model_outputs("multi")
planning_root = DRIVE_ROOT / "matched_planning"
run(["src.mylonas_ijepa_wm.run_wall_planning_matrix",
     "--matched", "--fixed-manifest", MODEL_DIRS["fixed"] / "episode_manifest.json",
     "--episode-manifest", MODEL_DIRS["multi"] / "episode_manifest.json",
     "--env-config", MODEL_DIRS["multi"] / "environment.json",
     "--encoder-checkpoint", ENCODER_LOCAL,
     "--fixed-checkpoint", MODEL_DIRS["fixed"] / "world_model/best.pt",
     "--multi-checkpoint", MODEL_DIRS["multi"] / "world_model/best.pt",
     "--output-root", planning_root,
     "--base-seed", "10000", "--num-episodes", "50",
     "--device", "cuda", "--precision", "auto", "--resume"])
summary = json.loads((planning_root / "matrix_summary.json").read_text())
print("Paired comparison:", json.dumps(summary["paired_comparison"], indent=2))
for row in summary["runs"]:
    print(row["model"], row["wall_layout"],
          f"{row['successes']}/{row['episodes']} successes")
print("All checkpoints and results:", DRIVE_ROOT)
